[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yy726/GPQA-eval/blob/submission-yy726/notebooks/11_sft_grpo_unsloth.ipynb)

# 11 · SFT + GRPO on Qwen3 Base（Unsloth + W&B）

结构参考 Unsloth 官方 Colab [Qwen3 (4B) Instruct](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_(4B)-Instruct.ipynb) /
[Qwen3 (4B) GRPO](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_(4B)-GRPO.ipynb)：同一个模型开 `fast_inference=True`（内置 vLLM），
先 **SFT**，再 **GRPO**；base / SFT / GRPO 各做一次 GPQA 快速评测（只输出聚合数字），W&B 记录曲线。

**X1 之后的改动**（详见 `experiments/results.md`）：
1. **结束符**：照 OLMo 3（open-instruct）的做法，回答以 `<|endoftext|>` 结束（`CHAT_STYLE="olmo"`）。X1 里 SFT 后模型不会写 `<|im_end|>`（Qwen3 Base 没学过这个 token，LoRA 也改不了），而是写一个乱码 token `\tNdrFc` 然后继续编下一题，截断率 97%。
2. **长度**：锁定评测的生成预算改为 **8192**；快速评测用满 8192，GRPO 回答上限 6144，截断惩罚改成 DAPO 的"软超长惩罚"。
3. **SFT 数据**：去掉逻辑题；可选按 4-gram 覆盖度挑数据（`SFT_SELECT="coverage"`，Dr.SCI 的 EESFT）。
4. **GRPO**：SciQ 太简单（X1 结束时 82% 的组没有信号）→ MMLU-Pro STEM 选择题 + Dr.SCI 数值题；训练前按通过率预筛、两段课程；每步 4 道题 × 8 个回答；`epsilon_high=0.28`。

**推荐流程（照 OLMo 3：SFT → DPO → RL）**：
本 notebook 跑 SFT 并设 `PUSH_SFT_REPO` → notebook 12 用 Dolci-Instruct-DPO 做 DPO 并推到 HF → 回到本 notebook，`RUN_SFT=False`、`INIT_MODEL`=DPO 模型，只跑 GRPO。

| 模式 | 用途 | SFT | GRPO | 快速评测 |
|---|---|---|---|---|
| `quick` | 验证流程（T4 也行，0.6B） | 200 + 200 条，40 步 | 30 步 × 2 题，回答 ≤512 | Diamond 前 60 题，≤1024 |
| `medium` | 1.7B 看变化 | 1000 + 500 条 | 60 步 × 4 题，≤2048 | Diamond 198 题，≤4096 |
| `a100` | **4B 正式实验**（A100 40GB） | 2 万 + 2000 条，≤6144 token | 100 步 × 4 题 × 8，≤6144 | Diamond + Main，≤8192 |

⚠️ 快速评测**不是**排行榜分数（单样本、选项顺序与 lm-eval 不同）。正式分数：合并推到 HF 后用 `main` 分支的 `01_eval_gpqa.ipynb`。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# 克隆提交分支（数据准备 / 去污染 / 共用代码 scripts/recipe_utils.py 都在里面）
import os, sys
if not os.path.exists("GPQA-eval"):
    !git clone -q -b submission-yy726 https://github.com/yy726/GPQA-eval.git
%cd GPQA-eval
!git pull -q
sys.path.insert(0, os.path.abspath("scripts"))

In [ ]:
# 安装（约 3–5 分钟）。输出不隐藏：如果安装报错请看这里。
# 2026-11 实测可用：unsloth 2026.11.3、vllm 0.31、trl 1.15、transformers 5.17（A100）
!pip install --upgrade -q uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
!uv pip install unsloth vllm wandb {_numpy} {_pil}

In [ ]:
import os, importlib.metadata as md, torch
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"   # Unsloth：GRPO 时让 vLLM 和训练共享显存
if torch.cuda.get_device_capability()[0] < 8:   # T4 不支持 FlashInfer 采样器
    os.environ["UNSLOTH_VLLM_NO_FLASHINFER"] = "1"
    os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
    print("T4 detected: FlashInfer disabled for vLLM")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM_GB = round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)
VERSIONS = {}
for p in ["unsloth", "vllm", "trl", "transformers", "torch", "peft", "wandb"]:
    try: VERSIONS[p] = md.version(p)
    except md.PackageNotFoundError: VERSIONS[p] = None
print(GPU_NAME, GPU_MEM_GB, "GB"); print(VERSIONS)

In [ ]:
# Colab 左侧 🔑 Secrets：HF_TOKEN（必需；先在 https://huggingface.co/datasets/Idavidrein/gpqa 接受条款）
#                      WANDB_API_KEY（强烈建议；没有就不记录 W&B）
for _k in ["HF_TOKEN", "WANDB_API_KEY"]:
    try:
        from google.colab import userdata
        _v = userdata.get(_k)
        if _v:
            os.environ[_k] = _v
    except Exception:
        pass
from huggingface_hub import login
login(token = os.environ.get("HF_TOKEN"))
print("W&B key found" if os.environ.get("WANDB_API_KEY") else "No WANDB_API_KEY: W&B logging disabled")

## 0. 配置

In [ ]:
EXP_NAME     = "X1b"                          # 实验编号：W&B run 名、报告里用
MODE         = "a100"                         # "quick" | "medium" | "a100"
MODEL_NAME   = "unsloth/Qwen3-4B-Base"        # quick: "unsloth/Qwen3-0.6B-Base"；medium: "unsloth/Qwen3-1.7B-Base"
BASE_MODEL   = "Qwen/Qwen3-4B-Base"           # 提交时填的原始 base
LOAD_IN_4BIT = False
CHAT_STYLE   = "olmo"   # "olmo"（回答以 <|endoftext|> 结束，推荐）| "plain"（无特殊 token）| "chatml"（X1 的旧模板，只作对比）

# 流程开关
RUN_SFT         = True
INIT_MODEL      = None   # 例如 notebook 12 推到 HF 的 DPO 模型；配合 RUN_SFT=False 只跑 GRPO
PUSH_SFT_REPO   = None   # 例如 "YOUR_HF_USERNAME/qwen3-4b-x1b-sft"：SFT 后合并推到 HF 私有仓库（notebook 12 从它开始）
RUN_GRPO        = True
PUSH_FINAL_REPO = None   # 例如 "YOUR_HF_USERNAME/qwen3-4b-x1b"：最后合并推到 HF（正式评测要公开仓库）

# SFT 数据（--include/--exclude 同时匹配来源名和 domain；X1 里 Logic Puzzles 占 29%，现在去掉）
DOLCI_DATASET    = "allenai/Dolci-Instruct-SFT-7B"
DOLCI_INCLUDE    = ["science", "math", "gsm"]
DOLCI_EXCLUDE    = ["safety", "wildguard", "python", "code", "aya", "precise", "flan", "logic"]
THINK_DATASET    = "allenai/Dolci-Think-SFT-7B"
THINK_N          = 0            # >0：再从 Dolci-Think 抽这么多条思维链（长度 ≤ sft_max_tokens）
THINK_INCLUDE    = ["science", "math"]
EXTRA_SFT        = []           # 见 experiments/DATASETS.md，例如 [dict(dataset="MegaScience/TextbookReasoning", n=4000)]
SFT_SELECT       = "coverage"   # "random" | "coverage"（Dr.SCI 的 EESFT：按 4-gram 覆盖度从候选池里挑）
SFT_POOL_FACTOR  = 3            # coverage 时先抽 3 倍候选
COVERAGE_NORM    = "sqrt"       # "none" = 完全照论文（偏向长样本）；"sqrt" = 除以 √长度

# GRPO
GRPO_DATA         = "mix"       # "mix"（MMLU-Pro STEM 选择题 + Dr.SCI 数值题）| "mmlu_pro_stem" | "drsci" | "sciq"
GRPO_FILTER       = True        # 训练前每题采样 4 次，只留答对 1–3 次的（去掉太易 / 太难的题）
CURRICULUM_ROUNDS = 2           # 2 = 两段：前一半较易（Dr.SCI 通过率 0.5–0.875），后一半较难（0.25–0.375），第二段用更新后的模型重新预筛
DRSCI_SUBJECTS    = ["physics", "chemistry", "biology"]
GRPO_LR           = 5e-6
EPSILON_HIGH      = 0.28        # DAPO / OLMo 3 的 clip-higher
FORMAT_WEIGHT     = 0.5         # 格式分（正确性 +2 为主）
OVERLONG_CACHE    = 1024        # 软超长惩罚：最后 1024 token 里从 0 线性扣到 −1

WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "gpqa-hackathon")
WANDB_ENTITY  = os.environ.get("WANDB_ENTITY")

PRESETS = dict(
    quick = dict(max_seq_len=2048, gpu_util=0.7, lora_rank=32,
                 sft_dolci_n=200, sft_mcq_n=200, sft_max_tokens=1024, sft_steps=40, sft_lr=2e-4, sft_batch=4, sft_accum=2,
                 grpo_steps=30, grpo_prompts_per_step=2, num_gen=4, grpo_max_completion=512, filter_k=4, filter_max_tokens=512,
                 eval_configs=["gpqa_diamond"], eval_n=60, eval_max_tokens=1024, shuffle_buffer=1000),
    medium = dict(max_seq_len=8192, gpu_util=0.75, lora_rank=32,
                  sft_dolci_n=1000, sft_mcq_n=500, sft_max_tokens=4096, sft_steps=150, sft_lr=2e-4, sft_batch=4, sft_accum=2,
                  grpo_steps=60, grpo_prompts_per_step=4, num_gen=8, grpo_max_completion=2048, filter_k=4, filter_max_tokens=2048,
                  eval_configs=["gpqa_diamond"], eval_n=None, eval_max_tokens=4096, shuffle_buffer=5000),
    a100 = dict(max_seq_len=12288, gpu_util=0.8, lora_rank=64,
                sft_dolci_n=20000, sft_mcq_n=2000, sft_max_tokens=6144, sft_steps=-1, sft_lr=1e-4, sft_batch=4, sft_accum=4,
                grpo_steps=100, grpo_prompts_per_step=4, num_gen=8, grpo_max_completion=6144, filter_k=4, filter_max_tokens=3072,
                eval_configs=["gpqa_diamond", "gpqa_main"], eval_n=None, eval_max_tokens=8192, shuffle_buffer=20000),
)
Q = PRESETS[MODE]
MAX_SEQ_LEN, GPU_UTIL, LORA_RANK = Q["max_seq_len"], Q["gpu_util"], Q["lora_rank"]
print(EXP_NAME, MODE, INIT_MODEL or MODEL_NAME, CHAT_STYLE, Q)

## 1. 加载模型（`fast_inference=True` = 训练和生成共用一个 vLLM 引擎）

In [ ]:
import os, torch
if torch.cuda.get_device_capability()[0] < 8:   # 再设一次，防止跳过了前面的格
    os.environ["UNSLOTH_VLLM_NO_FLASHINFER"] = "1"
    os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
from unsloth import FastLanguageModel
from recipe_utils import (RESPONSE_PARTS, STOP_STRS, setup_chat, chat_prompt, build_gpqa,
                          score_vllm_outputs, log_eval, print_table, sciq_mcq, sciq_format_rows, load_mcq_rows,
                          drsci_numeric_row, decontam_rows, n_chat_tokens, correctness_score, format_score,
                          soft_overlong_penalty, is_correct_and_stopped, config_kwargs, wandb_init)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = INIT_MODEL or MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = LOAD_IN_4BIT,
    fast_inference = True,
    max_lora_rank = LORA_RANK,
    gpu_memory_utilization = GPU_UTIL,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = LORA_RANK * 2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
# 模板 + 结束符（照 OLMo 3：最后一个 assistant 回合以 eos=<|endoftext|> 结束；generation_config 两个结束符都停）
STOP_IDS = setup_chat(tokenizer, model, style = CHAT_STYLE)
EOS_ID = tokenizer.eos_token_id
_demo = tokenizer.apply_chat_template([{"role": "user", "content": "Hi"}, {"role": "assistant", "content": "Hello"}], tokenize = False)
print(repr(_demo))
assert _demo.endswith(tokenizer.eos_token), "最后一个回合应以 eos 结束"
print("eos:", tokenizer.eos_token, EOS_ID, "| stop ids:", STOP_IDS, "| pad:", tokenizer.pad_token)

RUN_NAME = f"{EXP_NAME}-{MODEL_NAME.split('/')[-1]}-{MODE}"
wandb, REPORT_TO = wandb_init(WANDB_PROJECT, RUN_NAME, dict(
    exp = EXP_NAME, mode = MODE, model = MODEL_NAME, init_model = INIT_MODEL, base_model = BASE_MODEL, chat_style = CHAT_STYLE,
    run_sft = RUN_SFT, run_grpo = RUN_GRPO, dolci = DOLCI_DATASET, dolci_include = DOLCI_INCLUDE, dolci_exclude = DOLCI_EXCLUDE,
    think_n = THINK_N, extra_sft = EXTRA_SFT, sft_select = SFT_SELECT, sft_pool_factor = SFT_POOL_FACTOR,
    coverage_norm = COVERAGE_NORM, grpo_data = GRPO_DATA, grpo_filter = GRPO_FILTER, curriculum_rounds = CURRICULUM_ROUNDS,
    grpo_lr = GRPO_LR, epsilon_high = EPSILON_HIGH, format_weight = FORMAT_WEIGHT, overlong_cache = OVERLONG_CACHE,
    gpu = GPU_NAME, gpu_mem_gb = GPU_MEM_GB, versions = VERSIONS, **Q), WANDB_ENTITY)

## 2. GPQA 快速评测 + 诊断（只输出聚合数字；不记录题目或模型输出）

`flex` = 输出中最后一个 `(A)`–`(D)`（排行榜指标），`strict` = 第一个 `The answer is (X)`。诊断指标：
`repeat_rate`（某 8 词片段重复 ≥4 次，打转）、`answer_then_more`（答案之后还写了 ≥30% 的内容）、`glitch_rate`（出现 X1 里的乱码 token `\tNdrFc`）。

In [ ]:
import random, json, statistics, time
from collections import Counter
from datasets import load_dataset
from vllm import SamplingParams

EVAL_SETS = {}
for cfg in Q["eval_configs"]:
    rows = build_gpqa(cfg, Q["eval_n"])
    texts = [chat_prompt(tokenizer, p) for p, _ in rows]
    longest = max(len(tokenizer(t, add_special_tokens = False)["input_ids"]) for t in texts)
    EVAL_SETS[cfg] = (texts, [g for _, g in rows], longest)
EVAL_MAX_TOKENS = min(Q["eval_max_tokens"], MAX_SEQ_LEN - max(v[2] for v in EVAL_SETS.values()) - 8)
if EVAL_MAX_TOKENS < Q["eval_max_tokens"]:
    print(f"[!] eval_max_tokens clamped to {EVAL_MAX_TOKENS} to fit MAX_SEQ_LEN={MAX_SEQ_LEN}")
RESULTS = {}

def quick_eval(tag, lora_request = None):
    sp = SamplingParams(temperature = 0.6, top_p = 0.95, max_tokens = EVAL_MAX_TOKENS, seed = 0,
                        stop_token_ids = STOP_IDS, stop = STOP_STRS)
    RESULTS[tag] = {}
    for cfg, (texts, golds, _) in EVAL_SETS.items():
        t0 = time.time()
        r = score_vllm_outputs(model.fast_generate(texts, sampling_params = sp, lora_request = lora_request), golds)
        r["minutes"] = round((time.time() - t0) / 60, 1)
        RESULTS[tag][cfg] = r
        print(tag, cfg, r)
    log_eval(wandb, RESULTS[tag], tag)

def diagnose(tag, lora_request = None, k = 3):
    # 看清楚答案后面生成了什么（SciQ 验证集，公开数据；不跳过特殊 token）
    val = load_dataset("allenai/sciq", split = "validation").select(range(k))
    prompts = [chat_prompt(tokenizer, sciq_mcq(ex, 10_000 + i)[0]) for i, ex in enumerate(val)]
    sp = SamplingParams(temperature = 0.0, max_tokens = 512, stop_token_ids = STOP_IDS, stop = STOP_STRS,
                        skip_special_tokens = False)
    for o in model.fast_generate(prompts, sampling_params = sp, lora_request = lora_request):
        c = o.outputs[0]
        print(f"--- [{tag}] finish={c.finish_reason} stop={c.stop_reason!r} tokens={len(c.token_ids)}")
        print(repr(c.text[-300:]))

def stop_check(rows, k = 16):
    # teacher forcing：模型在训练样本答案结尾预测 eos 的概率（应接近 1）；以及输出层几行的范数
    os.environ["UNSLOTH_RETURN_LOGITS"] = "1"
    im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
    glitch = tokenizer.convert_tokens_to_ids("ĉNdrFc")
    out = {}
    try:
        p_eos, p_im_end, p_glitch = [], [], []
        for r in rows[:k]:
            ids = tokenizer(tokenizer.apply_chat_template(r["messages"], tokenize = False), return_tensors = "pt",
                            add_special_tokens = False).input_ids[:, -2048:].to(model.device)
            if ids[0, -1].item() != EOS_ID:
                continue
            with torch.no_grad():
                probs = torch.softmax(model(input_ids = ids[:, :-1]).logits[0, -1].float(), -1)
            p_eos.append(probs[EOS_ID].item()); p_im_end.append(probs[im_end].item()); p_glitch.append(probs[glitch].item())
        out = dict(n = len(p_eos), p_eos_mean = round(statistics.fmean(p_eos), 4) if p_eos else None,
                   p_eos_median = round(statistics.median(p_eos), 4) if p_eos else None,
                   p_im_end_mean = round(statistics.fmean(p_im_end), 4) if p_im_end else None,
                   p_glitch_mean = round(statistics.fmean(p_glitch), 4) if p_glitch else None)
        W = model.get_output_embeddings().weight
        norms = W.float().norm(dim = -1)
        out.update(norm_eos = round(norms[EOS_ID].item(), 3), norm_im_end = round(norms[im_end].item(), 3),
                   norm_glitch = round(norms[glitch].item(), 3), norm_mean = round(norms.mean().item(), 3))
    except Exception as e:
        print("stop check skipped:", type(e).__name__, e)
    finally:
        os.environ.pop("UNSLOTH_RETURN_LOGITS", None)
    print("stop check:", out)
    return out

quick_eval("0_init" if INIT_MODEL else "0_base")

## 3. SFT（chat 模板转成 `text` + `train_on_responses_only`）

1. **Dolci-Instruct** 按来源 / domain 过滤到科学、数学（不要逻辑题），长度 ≤ `sft_max_tokens`。
2. 可选 **Dolci-Think** 思维链（`THINK_N`）和 **额外数据源**（`EXTRA_SFT`）。
3. **覆盖度选择**（`SFT_SELECT="coverage"`）：先抽 `SFT_POOL_FACTOR` 倍候选，再按 Dr.SCI 的 EESFT 贪心挑出最能扩大 4-gram 覆盖的样本。
4. **选择题格式样本**：SciQ 训练集套 GPQA 模板，答案 = `support` 段落 + `The answer is (X).`。

全部过一遍 GPQA 13-gram 去污染。

In [ ]:
os.makedirs("data", exist_ok = True)
N_DOLCI, SFT_MAX, SHUF = Q["sft_dolci_n"], Q["sft_max_tokens"], Q["shuffle_buffer"]
POOL = SFT_POOL_FACTOR if SFT_SELECT == "coverage" else 1
inc = " ".join(DOLCI_INCLUDE); exc = " ".join(DOLCI_EXCLUDE)
inc_arg = f"--include {inc}" if inc else ""
N_POOL = N_DOLCI * POOL
if RUN_SFT:
    !python scripts/prepare_dolci_sft.py --dataset {DOLCI_DATASET} --out data/dolci.jsonl --n {N_POOL} --tokenizer {MODEL_NAME} --max-tokens {SFT_MAX} --shuffle-buffer {SHUF} {inc_arg} --exclude {exc}
    if THINK_N > 0:
        t_inc = " ".join(THINK_INCLUDE); N_THINK_POOL = THINK_N * POOL
        !python scripts/prepare_dolci_sft.py --dataset {THINK_DATASET} --out data/think.jsonl --n {N_THINK_POOL} --tokenizer {MODEL_NAME} --max-tokens {SFT_MAX} --shuffle-buffer {SHUF} --include {t_inc} --exclude {exc}

In [ ]:
DATA = {}
if RUN_SFT:
    from recipe_utils import prepare_sft_sources
    from select_coverage import select as coverage_select, hashed_coverage

    def _load(path):
        return [json.loads(l) for l in open(path)] if os.path.exists(path) else []

    pool = [{"messages": r["prompt"] + r["completion"], "source": r["source"], "n_tokens": r["n_tokens"], "part": "dolci"}
            for r in _load("data/dolci.jsonl")]
    pool += [{"messages": r["prompt"] + r["completion"], "source": r["source"], "n_tokens": r["n_tokens"], "part": "think"}
             for r in (_load("data/think.jsonl") if THINK_N > 0 else [])]
    if EXTRA_SFT:
        pool += [dict(r, part = "extra") for r in prepare_sft_sources(EXTRA_SFT, MODEL_NAME, SFT_MAX, scale = POOL, shuffle_buffer = SHUF)]
    pool, dropped = decontam_rows(pool, lambda r: {"messages": r["messages"]}, "sft_pool")
    target = N_DOLCI + (THINK_N if THINK_N > 0 else 0) + sum(s["n"] for s in EXTRA_SFT)
    texts = ["\n".join(m["content"] for m in r["messages"]) for r in pool]
    if SFT_SELECT == "coverage" and len(pool) > target:
        t0 = time.time()
        idx = coverage_select(texts, target, norm = COVERAGE_NORM)
        rand = random.Random(0).sample(range(len(pool)), min(target, len(pool)))
        cov = dict(selected_4grams = hashed_coverage([texts[i] for i in idx]),
                   random_4grams = hashed_coverage([texts[i] for i in rand]), minutes = round((time.time() - t0) / 60, 1))
        print("coverage selection:", cov)
    else:
        idx, cov = list(range(min(target, len(pool)))), {}
    chosen = [pool[i] for i in idx]
    sft_mcq = sciq_format_rows(Q["sft_mcq_n"])
    sft_rows = [{"messages": r["messages"]} for r in chosen] + sft_mcq
    random.Random(0).shuffle(sft_rows)
    DATA = dict(pool = len(pool), selected = len(chosen), sciq_format = len(sft_mcq), dropped_decontam = dropped,
                select = SFT_SELECT, coverage = cov, tokens = sum(r["n_tokens"] or 0 for r in chosen),
                by_part = Counter(r["part"] for r in chosen).most_common(),
                sources = Counter(r["source"] for r in chosen).most_common(10))
    wandb.config.update({"data": DATA}, allow_val_change = True)
    print(DATA)

In [ ]:
SFT, STOP = {}, {}
if RUN_SFT:
    from datasets import Dataset
    from trl import SFTTrainer, SFTConfig
    from unsloth.chat_templates import train_on_responses_only

    # 文本以 eos 结束（模板最后一个回合写的就是 eos），TRL 不会再额外追加
    sft_ds = Dataset.from_list([{"text": tokenizer.apply_chat_template(r["messages"], tokenize = False)} for r in sft_rows])
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = sft_ds,
        args = SFTConfig(
            dataset_text_field = "text",
            max_length = MAX_SEQ_LEN,
            per_device_train_batch_size = Q["sft_batch"],
            gradient_accumulation_steps = Q["sft_accum"],
            warmup_steps = 10,
            max_steps = Q["sft_steps"],
            num_train_epochs = 1,
            learning_rate = Q["sft_lr"],
            logging_steps = 5,
            optim = "adamw_8bit",
            weight_decay = 0.001,
            lr_scheduler_type = "linear",
            seed = 3407,
            report_to = REPORT_TO,
            run_name = RUN_NAME + "-sft",
            output_dir = "outputs/sft",
            save_strategy = "no",
        ),
    )
    instr_part, resp_part = RESPONSE_PARTS[CHAT_STYLE]
    trainer = train_on_responses_only(trainer, instruction_part = instr_part, response_part = resp_part)
    # 诊断：训练标签的最后一个 token 应该是 eos（模型要学会写它）
    try:
        _labels = [t for t in trainer.train_dataset[0]["labels"] if t != -100]
        print("label tail:", repr(tokenizer.decode(_labels[-12:])), "| ends with eos:", _labels[-1] == EOS_ID)
    except Exception as e:
        print("label check skipped:", e)

    sft_stats = trainer.train()
    sft_losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    SFT = dict(steps = trainer.state.global_step, minutes = round(sft_stats.metrics["train_runtime"] / 60, 1),
               loss_first = sft_losses[0] if sft_losses else None, loss_last = sft_losses[-1] if sft_losses else None)
    print(SFT)
    STOP = stop_check(sft_rows)
    model.save_lora("sft_lora")
    quick_eval("1_sft", model.load_lora("sft_lora"))
    diagnose("1_sft", model.load_lora("sft_lora"))
    if PUSH_SFT_REPO:
        try:
            model.push_to_hub_merged(PUSH_SFT_REPO, tokenizer, save_method = "merged_16bit",
                                     private = True, token = os.environ.get("HF_TOKEN"))
        except TypeError:   # 旧版 Unsloth 没有 private 参数：先在 HF 网页上建好私有仓库再推
            model.push_to_hub_merged(PUSH_SFT_REPO, tokenizer, save_method = "merged_16bit",
                                     token = os.environ.get("HF_TOKEN"))
        print("SFT pushed to", PUSH_SFT_REPO, "→ notebook 12 的 INIT_MODEL 填它")
    del sft_ds, trainer
    torch.cuda.empty_cache(); import gc; gc.collect()

## 4. GRPO（`vllm_sampling_params` + 多个奖励函数）

**题库**（`GRPO_DATA`）：
- `mix`（默认）：一半 **MMLU-Pro STEM**（物理/化学/生物/工程/数学，抽成 4 选 1，套 GPQA 模板，答案看最后一个 `(A)`–`(D)`），
  一半 **Dr.SCI 数值题**（[MiniByte-666/Dr.SCI](https://huggingface.co/datasets/MiniByte-666/Dr.SCI) 的可验证题里参考答案是数字的；用它自带的 `\boxed{}` 提示，相对误差 ≤1% 算对；没有 25% 的猜中率）。
- 课程（`CURRICULUM_ROUNDS=2`）：第一段 Dr.SCI 用 Qwen3-32B 通过率 0.5–0.875 的题，第二段用 0.25–0.375 的题；每段开始前用**当前模型**每题采样 4 次，只留答对 1–3 次的（`GRPO_FILTER`），对应 OLMo 3 的"去掉全对/全错的组"和"通过率 ≥0.875 的题不再抽"。

**奖励**：
- `correctness_reward`：答对 +2，答错 0，找不到答案 −1
- `format_reward`：答案以规定形式恰好出现一次 +0.5（选择题 `The answer is (X)`，数值题 `\boxed{}`）
- `overlong_reward`：DAPO 软超长惩罚——长度超过 `上限 − 1024` 后线性扣到 −1，被截断 −1

In [ ]:
import numpy as np
from datasets import Dataset

STEPS_PER_STAGE = max(1, Q["grpo_steps"] // max(1, CURRICULUM_ROUNDS))
NEED_PER_STAGE = STEPS_PER_STAGE * Q["grpo_prompts_per_step"]
CAND_PER_STAGE = NEED_PER_STAGE * (3 if GRPO_FILTER else 1)

def _mcq_rows(source, n, skip = 0):
    return [{"prompt": [{"role": "user", "content": r["prompt"]}], "answer": r["answer"], "answer_type": "mcq",
             "source": r["source"]} for r in load_mcq_rows(source, n, skip = skip)]

DRSCI_RANGES = [(0.5, 0.875), (0.25, 0.375)] if CURRICULUM_ROUNDS > 1 else [(0.25, 0.875)]
_drsci_cache = {}
def _drsci_rows(n, rng, offset = 0):
    if "ds" not in _drsci_cache:
        _drsci_cache["ds"] = load_dataset("MiniByte-666/Dr.SCI", data_files = "Dr_SCI_verifiable.parquet",
                                          split = "train").shuffle(seed = 0)
    out = []
    for row in _drsci_cache["ds"].select(range(offset, len(_drsci_cache["ds"]))):
        r = drsci_numeric_row(row, subjects = DRSCI_SUBJECTS, difficulty = rng)
        if r:
            out.append(r)
            if len(out) >= n:
                break
    return out

def stage_pool(stage):
    n = CAND_PER_STAGE
    if GRPO_DATA == "mix":
        rows = _mcq_rows("mmlu_pro_stem", n // 2 * (stage + 1))[-(n // 2):] + _drsci_rows(n - n // 2, DRSCI_RANGES[min(stage, len(DRSCI_RANGES) - 1)])
    elif GRPO_DATA == "drsci":
        rows = _drsci_rows(n, DRSCI_RANGES[min(stage, len(DRSCI_RANGES) - 1)])
    elif GRPO_DATA == "sciq":
        rows = _mcq_rows("sciq", n * (stage + 1), skip = 4000)[-n:]      # SciQ 前段用于 SFT 格式样本
    else:
        rows = _mcq_rows("mmlu_pro_stem", n * (stage + 1))[-n:]
    rows, dropped = decontam_rows(rows, lambda r: {"prompt": r["prompt"]}, f"grpo_pool_s{stage}")
    random.Random(stage).shuffle(rows)
    return rows

def pass_filter(rows, lora_request, need):
    # 当前模型每题采样 k 次，只留答对 1..k-1 次的题（全对 / 全错的组没有学习信号）
    k = Q["filter_k"]
    sp = SamplingParams(n = k, temperature = 1.0, top_p = 1.0, max_tokens = Q["filter_max_tokens"], seed = 7,
                        stop_token_ids = STOP_IDS, stop = STOP_STRS)
    t0 = time.time()
    outs = model.fast_generate([chat_prompt(tokenizer, r["prompt"][0]["content"]) for r in rows],
                               sampling_params = sp, lora_request = lora_request)
    keep, hist = [], Counter()
    for r, o in zip(rows, outs):
        c = sum(is_correct_and_stopped(x.text, r["answer"], x.finish_reason, r["answer_type"]) for x in o.outputs)
        hist[c] += 1
        if 0 < c < k:
            keep.append(r)
    info = dict(candidates = len(rows), kept = len(keep), pass_hist = dict(sorted(hist.items())),
                minutes = round((time.time() - t0) / 60, 1))
    print("pass filter:", info)
    if len(keep) < need:
        print(f"[!] only {len(keep)} prompts passed the filter (need {need}); topping up with unfiltered ones")
        keep += [r for r in rows if r not in keep][: need - len(keep)]
    return keep[:need], info

# 题目长度上限（TRL 不支持 max_prompt_length，像 Unsloth 示例那样直接过滤）
MAX_PROMPT_LEN = 1024
MAX_COMPLETION = min(Q["grpo_max_completion"], MAX_SEQ_LEN - MAX_PROMPT_LEN)

def _fits(r):
    return n_chat_tokens(tokenizer, r["prompt"], add_generation_prompt = True) <= MAX_PROMPT_LEN

def format_reward(completions, answer_type, **kwargs):
    return [format_score(c[0]["content"], t, FORMAT_WEIGHT) for c, t in zip(completions, answer_type)]

def correctness_reward(completions, answer, answer_type, **kwargs):
    return [correctness_score(c[0]["content"], a, t) for c, a, t in zip(completions, answer, answer_type)]

def overlong_reward(completions, completion_ids = None, **kwargs):
    if completion_ids is None:
        completion_ids = [tokenizer(c[0]["content"], add_special_tokens = False)["input_ids"] for c in completions]
    return [soft_overlong_penalty(len(ids), MAX_COMPLETION, OVERLONG_CACHE) for ids in completion_ids]

print(dict(stages = CURRICULUM_ROUNDS, steps_per_stage = STEPS_PER_STAGE, prompts_per_stage = NEED_PER_STAGE,
           candidates_per_stage = CAND_PER_STAGE, max_completion = MAX_COMPLETION))

In [ ]:
GRPO = {}
if RUN_GRPO:
    from trl import GRPOConfig, GRPOTrainer
    vllm_sampling_params = SamplingParams(min_p = 0.1, top_p = 1.0, top_k = -1, seed = 3407,
                                          stop_token_ids = STOP_IDS, stop = STOP_STRS)
    NUM_GEN = Q["num_gen"]
    lora_req = model.load_lora("sft_lora") if RUN_SFT else None
    stages, log_all = [], []

    def _avg(log, key, part):
        vals = [h[key] for h in log if key in h]
        k = max(1, len(vals) // 5)
        return round(float(np.mean(vals[:k] if part == "first" else vals[-k:])), 3) if vals else None

    for stage in range(CURRICULUM_ROUNDS):
        rows = [r for r in stage_pool(stage) if _fits(r)]
        filt = {}
        if GRPO_FILTER:
            rows, filt = pass_filter(rows, lora_req, NEED_PER_STAGE)
        else:
            rows = rows[:NEED_PER_STAGE]
        ds = Dataset.from_list([{k: r[k] for k in ("prompt", "answer", "answer_type")} for r in rows])
        args = GRPOConfig(**config_kwargs(
            GRPOConfig,
            vllm_sampling_params = vllm_sampling_params,
            generation_kwargs = {"stop_token_ids": STOP_IDS},
            temperature = 1.0,
            learning_rate = GRPO_LR,
            weight_decay = 0.001,
            warmup_steps = max(1, STEPS_PER_STAGE // 10) if stage == 0 else 0,
            lr_scheduler_type = "constant_with_warmup",      # OLMo 3 RL：常数学习率
            optim = "adamw_8bit",
            logging_steps = 1,
            per_device_train_batch_size = NUM_GEN,            # 每步 = grpo_prompts_per_step 道题 × NUM_GEN 个回答
            gradient_accumulation_steps = Q["grpo_prompts_per_step"],
            num_generations = NUM_GEN,
            max_completion_length = MAX_COMPLETION,
            epsilon_high = EPSILON_HIGH,                      # clip-higher（DAPO / OLMo 3）；loss_type/beta 用 Unsloth 默认 dapo、0
            max_steps = STEPS_PER_STAGE,
            save_steps = 10_000,
            log_completions = False,                          # 不往 W&B 上传生成内容
            report_to = REPORT_TO,
            run_name = f"{RUN_NAME}-grpo-s{stage}",
            output_dir = f"outputs/grpo_s{stage}",
        ))
        trainer = GRPOTrainer(model = model, processing_class = tokenizer,
                              reward_funcs = [format_reward, correctness_reward, overlong_reward],
                              args = args, train_dataset = ds)
        stats = trainer.train()
        log = trainer.state.log_history
        s = dict(stage = stage, steps = trainer.state.global_step, minutes = round(stats.metrics["train_runtime"] / 60, 1),
                 prompts = len(ds), by_type = dict(Counter(r["answer_type"] for r in rows)), filter = filt)
        for key in ["reward", "rewards/format_reward/mean", "rewards/correctness_reward/mean", "rewards/overlong_reward/mean",
                    "completions/mean_length", "completions/clipped_ratio", "frac_reward_zero_std"]:
            if any(key in h for h in log):
                s[key] = {"first": _avg(log, key, "first"), "last": _avg(log, key, "last")}
        stages.append(s)
        print(s)
        model.save_lora(f"grpo_lora_s{stage}")
        lora_req = model.load_lora(f"grpo_lora_s{stage}")
        del trainer, ds
        torch.cuda.empty_cache()
    GRPO = dict(stages = stages, max_completion = MAX_COMPLETION, grpo_data = GRPO_DATA)
    quick_eval("2_grpo", lora_req)
    diagnose("2_grpo", lora_req)

## 5. 汇总（W&B 上也有同样的表；最后打印的 JSON 可以贴回给 Claude）

In [ ]:
rows = print_table(RESULTS)
print("SFT:", SFT); print("STOP:", STOP); print("GRPO:", GRPO)
REPORT = dict(notebook = "11_sft_grpo_unsloth", exp = EXP_NAME, mode = MODE, model = MODEL_NAME, init_model = INIT_MODEL,
              chat_style = CHAT_STYLE, gpu = GPU_NAME, gpu_mem_gb = GPU_MEM_GB, q = Q, dolci = DOLCI_DATASET,
              dolci_include = DOLCI_INCLUDE, think_n = THINK_N, extra_sft = EXTRA_SFT, sft_select = SFT_SELECT,
              grpo_data = GRPO_DATA, grpo_filter = GRPO_FILTER, curriculum_rounds = CURRICULUM_ROUNDS, grpo_lr = GRPO_LR,
              epsilon_high = EPSILON_HIGH, data = DATA, sft = SFT, stop_check = STOP, grpo = GRPO, quick_eval = RESULTS,
              versions = VERSIONS, wandb = wandb.run.url if REPORT_TO == "wandb" else None)
os.makedirs("outputs", exist_ok = True)
json.dump(REPORT, open(f"outputs/report_{EXP_NAME}.json", "w"), indent = 1, default = str)
if REPORT_TO == "wandb":
    wandb.log({"gpqa_quick_eval": wandb.Table(columns = ["stage", "set", "acc_flex", "acc_strict", "extract_rate",
                                                          "truncation_rate", "mean_tokens"], data = rows)})
    art = wandb.Artifact(f"report-{RUN_NAME}", type = "report")
    art.add_file(f"outputs/report_{EXP_NAME}.json")
    wandb.log_artifact(art)
    wandb.finish()
print("=" * 30, "把下面整段 JSON 贴回给 Claude", "=" * 30)
print(json.dumps(REPORT, ensure_ascii = False, default = str))

## 6. 合并保存 → 推到 Hub → 用 `main` 分支的 `01` 做正式评测

`merged_16bit` 把 LoRA 合并进权重（排行榜要求）；保存的 tokenizer 带着本 notebook 的模板和 eos=`<|endoftext|>`，`generation_config` 里两个结束符都在，锁定评测会正确停止。
推到 Hub 后**换一个新的 runtime** 打开 [`01_eval_gpqa.ipynb`](https://colab.research.google.com/github/yy726/GPQA-eval/blob/main/notebooks/01_eval_gpqa.ipynb)（生成预算 8192）。A100 上训练的模型提交时在 `notes` 里注明 A100（不参与排名）。

In [ ]:
if PUSH_FINAL_REPO:
    model.push_to_hub_merged(PUSH_FINAL_REPO, tokenizer, save_method = "merged_16bit", token = os.environ.get("HF_TOKEN"))
    print(f"pushed: https://huggingface.co/{PUSH_FINAL_REPO}")
else:
    model.save_pretrained_merged("outputs/merged_16bit", tokenizer, save_method = "merged_16bit")
    print("saved locally: outputs/merged_16bit（设 PUSH_FINAL_REPO 可以直接推到 HF）")